# Notebook B: is the count signal actually *used*? Patching and steering LLaVA (GPU, about 1-1.5 h)

Notebook A showed the model builds a clean linear count signal at layers 8-16 and that the grating signal is about half as strong from layer 14 on when history is in the prompt. It cannot show whether the model *uses* the count signal. Here we intervene on the residual stream at the answer position (last prompt token).

**B1. Patch history state into a no-history run** (same image). At layer $L$ we overwrite the no-history residual stream at the last position with the cached history-run one. *Recovery* at layer $L$ is the fraction of the history-induced shift in $\Delta$ that comes back. It tells us from which layer on the count information sits at the answer position (if recovery is already ~1, later layers no longer need the history tokens).
**B1r. Reverse:** put the no-history state into a history run; the fraction of the shift *removed*.
**B2. Steer** a no-history run by adding $\pm$ a direction (norm matched to half the natural high-count minus low-count jump) at layer $L$. Directions: the *shift* direction (mean history-minus-no-history difference, high vs low count), the *count* direction from A, the grating-side *evidence* direction (positive control: should move $\Delta$ a lot), a *shuffled-label* direction and a *random* direction (negative controls).
**B0. Where does the evidence loss come from?** Before the interventions: swing under five prompts (none, balanced 5-of-10 count, short count, a neutral sentence with no trial information, count placed before the image). In the main run the loss was ~0.4-0.5 at every count, even balanced, so it is probably a cost of extra text, not of the prior.
**C. Interference.** In the same runs: does steering or patching also change the **swing** (right-grating minus left-grating $\Delta$)? If the count direction is also what weakens the evidence, adding it to a no-history run should shrink the swing.

**Reading the result.** At late layers the count and evidence directions have cosine ~0.7 (both point along the answer axis), so a steering effect there is not diagnostic. The informative layers are 8-12, where cosine is ~0: if adding the count direction there moves $\Delta$, later layers read it.

**Setup:** GPU on, Internet on (model download), dataset `llava-pve-outputs-run` attached. It recomputes the trials from `SEED` and the directions from the cache, so no other files are needed. Every stage checkpoints to `/kaggle/working/B_out` and resumes if the session dies; use *Save & Run All (Commit)*.
**This notebook was tested on a mock model, not on the real LLaVA. The self-test cell stops with a clear message if a hook is wrong.**

What it does:

It regenerates the same 600 trials from SEED=0 and recomputes the directions from the cache, so it needs no other files.
Self-tests run first. They check that it reproduces the cached Δ, that a zero edit does nothing, that the hook state matches hidden_states[L] (catching an off-by-one layer), and that steering along the evidence direction moves Δ. It stops with a message if any fail.
B1: it patches the history-run state at the answer position into the no-history run, and the reverse, at layers 4–28. This gives the fraction of the shift that returns or is erased.
B2: it steers the no-history run with the shift, count, evidence, shuffled-label and two random directions at layers 8–20. All have the same norm, equal to half the natural high-vs-low-count jump.
C: it measures whether patching or steering also changes the grating swing.
How to read B
Layers 8–12: the count and evidence directions are nearly orthogonal there. If the count direction moves Δ clearly more than the random and shuffled directions at those layers, later layers read it, which is the first causal evidence we’d have.
Layers 16+: both directions point along the answer axis (cosine about 0.7), so an effect there doesn’t show the model uses a prior. Treat it as a push toward “right”.
Evidence direction: this is the positive control. If it doesn’t move Δ strongly, something is wrong with the steering.
Swing under patching: if patching the history state also cuts the swing to about 0.55, the last-position state carries the interference, not only the prior.

There are two things I can’t predict. The 4-bit model may not reproduce the cached Δ to within my 0.35 tolerance, in which case the first self-test stops and tells you. And a layer-indexing mistake would be caught by the cosine check against the cached hidden state, not before.

## What the output means

The strongest result is that the suppression is much bigger than I told you, it starts much earlier, and it has little to do with the prior.

**1. The earlier numbers were confounded, as I suspected.**
- Measured within each count value, the evidence signal in the history run is about 0.3–0.4 of the no-history run from layer 10 onward. My earlier 0.53 was too high.
- The layer-12 outlier (1.79) is gone. It came from side and count being correlated, so the count signal leaked into the right-minus-left difference.

**2. It is gradual and starts at layer 1, not at layers 12–14.**
- `axis_ratio` is 0.72–0.79 at layers 1–4, then falls from 0.69 at layer 5 to 0.37 at layer 10, and stays about 0.3–0.4 after that.
- At layers 6–9 the count direction barely exists yet (correlation 0.08 at layer 6, 0.32 at layer 8), yet the evidence is already down to 0.5–0.6. So the count arriving can't be what suppresses the evidence.
- My earlier claim that the suppression begins where the count arrives is retracted.

**3. It doesn't depend on the count at all.** I checked this in the behavior data we already have, with no new run. Swing is the right-minus-left Δ at contrast ≥ 0.09, history run over no-history run:

| count in prompt | swing ratio |
|---|---|
| ≤2 right | 0.50 |
| 3–4 | 0.48 |
| **5 (balanced)** | **0.42** |
| 6–7 | 0.46 |
| ≥8 | 0.40 |

A balanced 5-of-10 sentence carries no prior and still cuts the swing by more than half. So the evidence loss is a cost of having that sentence in the prompt, not a conflict between prior and evidence. If anything it is slightly worse at extreme counts (0.40 vs 0.42–0.50), but the intervals overlap, so I wouldn't read anything into that.

**4. The residual-stream ratio (0.3–0.4) is lower than the behavioral swing ratio (0.4–0.5).** The logit readout is probably nonlinear or saturating, so the two shouldn't be equated one-to-one.

## What this changes

- "Prior vs evidence interference" is not a clean finding. It is a format cost, probably extra tokens, longer distance, or diluted attention to the image. The prior effect (+0.75 logits) is a separate, smaller thing sitting on top.
- For the mouse comparison, $\gamma/\beta$ computed inside the history condition (12% contrast shift, β = 6.3) is the one that matches how the prior and evidence are actually combined. The no-history β (5%) is the number that mixes two different prompts. I'd report 12% as primary and 5% as a bound, with the format cost stated.
- Test C in B (does the prior direction cut the swing) should now come out null. That wouldn't be a failure; it just agrees with this result.

## What to do now

I added a new cell to notebook B (re-attached, replacing the earlier one): **B0, "where does the evidence loss come from?"** It runs in about 5 minutes and measures the swing under five prompts: no history, a balanced 5-of-10 sentence, a short "Recent trials: 5 left, 5 right.", a neutral sentence of similar length with no trial information, and the count sentence placed before the image. That separates "any extra text hurts" from "this wording hurts" from "position hurts", and tells us whether a prompt can carry the prior without the cost.

1. Run B with `QUICK = True` first (about 5 min). If the self-tests pass, rerun with `QUICK = False`. The B0 table appears first in the output.
2. Paste back everything it prints.
3. D is still waiting on your IBL account.

If a shorter or reordered format keeps the prior effect and avoids the loss, we'd redo the main run with it and the comparison with the mouse gets cleaner. If nothing does, we report the cost and carry on with the 12% number.

In [1]:
# ===================== CONFIG =====================
MOCK = False
QUICK = True                # True = tiny smoke test (~5 min): fewer trials and layers
SEED = 0                     # must match the main run (the trials are regenerated from it)
DATA = "/kaggle/input/datasets/ponishbhatia3/llava-pve-outputs-run"
OUT = "/kaggle/working/B_out"
N_EXT, N_REST = 110, 50                      # patch trials: with extreme counts (<=2 or >=8 'right'), and the rest
N_STEER = 80                                 # steering images (balanced left/right, contrast >= 0.0625)
PATCH_LAYERS = [4, 8, 10, 12, 14, 16, 18, 20, 24, 28]
STEER_LAYERS = [8, 10, 12, 14, 16, 20]
DIRS = ["shift", "prior", "evid", "shuf", "rand1", "rand2"]
DO_REVERSE = True
if QUICK:
    N_EXT, N_REST, N_STEER, PATCH_LAYERS, STEER_LAYERS, DIRS = 24, 8, 16, [8, 12, 16], [10, 14], ["shift", "evid", "rand1"]
import os, re, glob, json, math, time, zipfile, shutil, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [3]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-b-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/B_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_EXT=N_EXT, N_REST=N_REST, N_STEER=N_STEER, PATCH_LAYERS=PATCH_LAYERS, STEER_LAYERS=STEER_LAYERS, DIRS=DIRS, DO_REVERSE=DO_REVERSE)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook B (patching/steering). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE B checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("patch_rows.csv", "steer_rows.csv"):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

[checkpoint] created dataset ponishbhatia3/llava-pve-b-ckpt (preflight) | server says: Ok
checkpoint push works; later pushes happen every 10 min and at the end of each stage


In [4]:
# ===================== trials, prompts, images: identical to the main run (counts10 history) =====================
CONTRASTS = [0.0, 0.03, 0.045, 0.0625, 0.09, 0.125, 1.0]; N_FULL, FULL_UNBIASED, WINDOW = 600, 90, 10
SIZE = 336; _Y, _X = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)
def make_trials(n, seed, contrasts, n_unbiased):
    rng = np.random.default_rng(seed); p = np.full(n, 0.5)
    t, cur = n_unbiased, float(rng.choice([0.2, 0.8]))
    while t < n:
        L = 20 + int(rng.exponential(60))
        while L > 100: L = 20 + int(rng.exponential(60))
        p[t:t + L] = cur; t, cur = t + L, round(1.0 - cur, 2)
    side = np.where(rng.random(n) < p, "right", "left"); c = rng.choice(contrasts, n)
    return dict(n=n, p_right=p, side=side, contrast=c, dx=rng.uniform(-15, 15, n), dy=rng.uniform(-30, 30, n), phi=rng.uniform(0, 2 * np.pi, n))
def render(T, t, seed):
    cx = SIZE / 2 + (84 if T["side"][t] == "right" else -84) + T["dx"][t]; cy = SIZE / 2 + T["dy"][t]
    env = np.exp(-((_X - cx) ** 2 + (_Y - cy) ** 2) / (2 * 36 ** 2))
    a = 0.5 + 0.5 * T["contrast"][t] * env * np.sin(2 * np.pi * (_X - cx) / 24 + T["phi"][t])
    a = a + np.random.default_rng(seed * 100003 + t).normal(0, 0.02, a.shape)
    return Image.fromarray(np.stack([(np.clip(a, 0, 1) * 255).astype(np.uint8)] * 3, -1))
def hist_counts(T, t, w):
    past = T["side"][max(0, t - w):t]; return len(past), int((past == "right").sum())
def history_text(T, t):
    n, nr = hist_counts(T, t, WINDOW)
    return "No previous trials." if n == 0 else f"In the last {n} trials the grating was on the left {n - nr} times and on the right {nr} times."
QS = ["Is the grating on the left or right of the image? Answer with one word.", "Is the grating on the right or left of the image? Answer with one word."]
def prompt(hist, q): return f"USER: <image>\n{hist}\n{q} ASSISTANT:" if hist else f"USER: <image>\n{q} ASSISTANT:"
T = make_trials(N_FULL, SEED + 1, CONTRASTS, FULL_UNBIASED)

In [5]:
# ===================== cache + directions (recomputed from the cached residual streams, same recipe as notebook A) =====================
ROWS = pd.read_csv(glob.glob(f"{DATA}/**/full_rows_all.csv", recursive=True)[0])
HS = np.concatenate([np.load(f, mmap_mode="r") for f in sorted(glob.glob(f"{DATA}/**/full_hs_*.npy", recursive=True))])    # (trial, [none,hist], layer 0..32, dim)
NTR, _, NLAY, D = HS.shape
w_ = ROWS.pivot(index="t", columns="order", values="delta")
M = ROWS[ROWS.order == "none"].set_index("t").loc[w_.index][["side", "contrast", "p_right", "n_hist", "n_hist_right"]].copy()
M["d_none"], M["d_hist"] = w_["none"], w_["hist"]; M["shift"] = M.d_hist - M.d_none
assert (M.side.values == T["side"]).all() and np.allclose(M.contrast.values, T["contrast"]), "regenerated trials differ from the main run: check SEED / CONTRASTS"
NR = M.n_hist_right.values.astype(int); FULL = (M.n_hist.values == 10)
frac = (NR + .5) / (M.n_hist.values + 1); TP = np.log(frac / (1 - frac))
cell = pd.get_dummies(M.side.astype(str) + "|" + M.contrast.astype(str), dtype=float).values
lsq = lambda A, B: np.linalg.lstsq(A, B, rcond=None)[0]
def count_dir(X, target, Cc):
    Xr = X - Cc @ lsq(Cc, X); yt = target - Cc @ lsq(Cc, target); d = Xr.T @ yt; return d / (np.linalg.norm(d) + 1e-12)
hi_c = M.contrast.values >= 0.09; sr = M.side.values == "right"
hi_cnt, lo_cnt = FULL & (NR >= 8), FULL & (NR <= 2)
rng0 = np.random.default_rng(0); perm = rng0.permutation(FULL.sum())
DIRVEC, NORM = {}, np.zeros(NLAY)
for name in DIRS: DIRVEC[name] = np.zeros((NLAY, D), np.float32)
for L in range(1, NLAY):
    Xn, Xh = HS[:, 0, L, :].astype(np.float32), HS[:, 1, L, :].astype(np.float32)
    shift = ((Xh - Xn)[hi_cnt].mean(0) - (Xh - Xn)[lo_cnt].mean(0)) / 2          # natural half-jump from a no-history state to a high/low-count state
    NORM[L] = np.linalg.norm(shift); unit = lambda v: v / (np.linalg.norm(v) + 1e-12)
    cand = {"shift": unit(shift),
            "prior": count_dir(Xh[FULL], TP[FULL], cell[FULL]),
            "evid": unit(Xn[hi_c & sr].mean(0) - Xn[hi_c & ~sr].mean(0)),
            "shuf": count_dir(Xh[FULL], TP[FULL][perm], cell[FULL]),
            "rand1": unit(np.random.default_rng(11 + L).normal(size=D)), "rand2": unit(np.random.default_rng(97 + L).normal(size=D))}
    for name in DIRS: DIRVEC[name][L] = cand[name] * NORM[L]                     # every direction gets the SAME norm
nat_hi = M["shift"][hi_cnt].mean(); nat_lo = M["shift"][lo_cnt].mean(); NAT_HALF = (nat_hi - nat_lo) / 2
print(f"cache {HS.shape} | natural behavioral half-jump in delta (high-count minus low-count)/2 = {NAT_HALF:.2f} logits")
print("steering norm per layer (matched across directions):", {L: round(float(NORM[L]), 1) for L in (8, 10, 12, 14, 16, 20)})
print("cos(prior, evid) at layers 8/12/16/20:", {L: round(float(DIRVEC['prior'][L] @ DIRVEC['evid'][L] / NORM[L] ** 2), 2) for L in (8, 12, 16, 20)} if "prior" in DIRS and "evid" in DIRS else "n/a")

cache (600, 2, 33, 4096) | natural behavioral half-jump in delta (high-count minus low-count)/2 = 0.33 logits
steering norm per layer (matched across directions): {8: 0.0, 10: 0.1, 12: 0.6, 14: 1.1, 16: 1.9, 20: 3.2}
cos(prior, evid) at layers 8/12/16/20: n/a


In [6]:
# ===================== model runner =====================
if MOCK:
    class Runner:
        """Numpy stand-in: state at layer L is linear in (evidence, count); delta reads it out. Pipeline test only."""
        def __init__(s): s.rng = np.random.default_rng(5)
        def prep(s, img, text):
            a = np.abs(np.asarray(img)[..., 0].astype(float) - 128); cx = (a.sum(0) * np.arange(SIZE)).sum() / (a.sum() + 1e-6)
            m = re.search(r"on the right (\d+) times", text); n = None
            if m: nr = int(m.group(1)); nl = int(re.search(r"on the left (\d+) times", text).group(1)); n = (nr + .5) / (nr + nl + 1)
            return dict(ev=float(np.clip((cx - 168) * 0.03, -1.7, 1.7)), frac=n)
        def state(s, inp, L):
            x = np.zeros(D, np.float32); h = np.random.default_rng(int(abs(inp["ev"]) * 1e4) % 99991)
            x += 0.02 * h.normal(size=D)
            if L >= 4: x[0] += inp["ev"] * (0.5 if inp["frac"] is not None else 1.0)
            if L >= 8 and inp["frac"] is not None: x[1] += (inp["frac"] - .5) * 3; x[0] += (inp["frac"] - .5) * 0.5
            return x
        def readout(s, x): return -0.7 + 3.0 * x[0] + 1.0 * x[1]
        def delta(s, inp, edit=None):
            x = s.state(inp, 32)
            if edit is not None:
                L, mode, vec = edit; xl = s.state(inp, L)
                x = x + ((vec - xl) if mode == "replace" else vec)
            return float(s.readout(x))
        def hs(s, inp): return np.stack([s.state(inp, L) for L in range(NLAY)]).astype(np.float16)
    RUN = Runner()
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto",
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    L_ID, R_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("Left", "Right")]
    _lists = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == 32]
    assert len(_lists) == 1, f"expected exactly one 32-layer list, found {[n for n, _ in _lists]}"
    LAYERS = _lists[0][1]; print("language-model layers found at:", _lists[0][0])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp, edit=None):
            handle = None
            if edit is not None:
                L, mode, vec = edit; v = torch.tensor(vec, dtype=torch.float16, device=model.device)
                def hook(mod, args, output):                      # output of decoder layer L-1 == hidden_states[L]
                    h = output[0] if isinstance(output, tuple) else output
                    h = h.clone(); h[:, -1, :] = v.to(h.dtype) if mode == "replace" else h[:, -1, :] + v.to(h.dtype)
                    return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
                handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally:
                if handle is not None: handle.remove()
            return float((lg[R_ID] - lg[L_ID]).item())
        def hs(s, inp):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            return torch.stack([h[0, -1] for h in out.hidden_states]).float().cpu().numpy().astype(np.float16)
        def captured(s, inp, L):                                  # what the hook sees at layer L (for the self-test)
            box = {}
            def hook(mod, args, output): box["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
            hd = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): model(**inp)
            finally: hd.remove()
            return box["h"]
    RUN = Runner()
print("runner ready")

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

language-model layers found at: model.language_model.layers
runner ready


In [7]:
# ===================== SELF-TEST: stop here if anything is off =====================
def inputs(t, with_hist, q=0):
    return RUN.prep(render(T, t, SEED + 1), prompt(history_text(T, t) if with_hist else None, QS[q]))
ok = np.where(FULL & (M.contrast.values >= 0.09))[0]; test_ts = list(ok[:6])
res = []
for t in test_ts:
    dn = (RUN.delta(inputs(t, False, 0)) + RUN.delta(inputs(t, False, 1))) / 2; dh = (RUN.delta(inputs(t, True, 0)) + RUN.delta(inputs(t, True, 1))) / 2
    res.append((dn - M.d_none.values[t], dh - M.d_hist.values[t]))
res = np.array(res); say(f"[self-test 1] reproduction of the cached delta (new - cached): none max|diff| {np.abs(res[:, 0]).max():.3f}, hist max|diff| {np.abs(res[:, 1]).max():.3f}")
assert np.abs(res).max() < 0.35, "cannot reproduce the cached deltas: trials, prompts or model setup differ from the main run"
t = test_ts[0]; inp = inputs(t, True, 0); d0 = RUN.delta(inp)
d_id = RUN.delta(inp, edit=(12, "add", np.zeros(D, np.float32))); say(f"[self-test 2] adding a zero vector changes delta by {abs(d_id - d0):.4f} (should be ~0)")
assert abs(d_id - d0) < 0.05
own = HS[t, 1, 12, :].astype(np.float32); d_own = RUN.delta(inp, edit=(12, "replace", own)); say(f"[self-test 3] replacing the state with the history run's own cached state changes delta by {abs(d_own - d0):.4f} (should be small)")
assert abs(d_own - d0) < 0.25
if not MOCK:
    cap = RUN.captured(inp, 12); cache = HS[t, 1, 12, :].astype(np.float32)
    cs = float(cap @ cache / (np.linalg.norm(cap) * np.linalg.norm(cache))); say(f"[self-test 4] hook state vs cached hidden_states[12]: cosine {cs:.4f} (should be ~1)")
    assert cs > 0.98, "hook indexing does not match hidden_states: layer off by one?"
big = RUN.delta(inp, edit=(12, "add", DIRVEC["evid"][12] * 3 if "evid" in DIRS else np.ones(D, np.float32) * 5)); say(f"[self-test 5] adding 3x the evidence direction at layer 12 moves delta by {big - d0:+.2f} (should be clearly positive)")
say("self-tests passed")

[self-test 1] reproduction of the cached delta (new - cached): none max|diff| 0.000, hist max|diff| 0.000
[self-test 2] adding a zero vector changes delta by 0.0000 (should be ~0)
[self-test 3] replacing the state with the history run's own cached state changes delta by 0.0000 (should be small)
[self-test 4] hook state vs cached hidden_states[12]: cosine 1.0000 (should be ~1)
[self-test 5] adding 3x the evidence direction at layer 12 moves delta by -0.02 (should be clearly positive)
self-tests passed


In [8]:
# ===================== B0: where does the evidence loss come from? (no hidden states, ~5 min) =====================
# In the main run the swing (right-minus-left delta) was ~0.4-0.5 of the no-history swing at EVERY count, even balanced 5-of-10.
# So the loss is not about the prior. Here: does any extra sentence between image and question cause it, or position, or the count wording?
rngb = np.random.default_rng(3); okb = np.where(M.contrast.values >= 0.09)[0]
B0_T = np.sort(np.r_[rngb.choice(okb[M.side.values[okb] == "right"], 60, replace=False), rngb.choice(okb[M.side.values[okb] == "left"], 60, replace=False)])
if QUICK: B0_T = B0_T[::6]
CNT5 = "In the last 10 trials the grating was on the left 5 times and on the right 5 times."
VAR = {"none": lambda q: prompt(None, q),
       "count_5_of_10": lambda q: prompt(CNT5, q),
       "short_count": lambda q: prompt("Recent trials: 5 left, 5 right.", q),
       "neutral_same_length": lambda q: prompt("The picture was taken with a plain camera in a quiet room on a sunny afternoon.", q),
       "count_before_image": lambda q: f"USER: {CNT5}\n<image>\n{q} ASSISTANT:"}
b0 = []
for t in B0_T:
    img = render(T, t, SEED + 1)
    for name, f in VAR.items(): b0.append(dict(t=t, side=M.side.values[t], variant=name, delta=RUN.delta(RUN.prep(img, f(QS[0])))))
B0 = pd.DataFrame(b0); B0.to_csv(f"{OUT}/b0_evidence_loss.csv", index=False); maybe_push('B0 done', force=True)
def _sw(df): return df[df.side == "right"].delta.mean() - df[df.side == "left"].delta.mean()
sw0 = _sw(B0[B0.variant == "none"]); say(f"\n[B0] swing (right minus left delta, contrast >= 0.09, {len(B0_T)} images) for different prompt variants; ratio to no-history ({sw0:.2f})")
for name in VAR:
    g = B0[B0.variant == name]; rb = np.random.default_rng(0); v = []
    for _ in range(300):
        pick = rb.integers(0, len(B0_T), len(B0_T)); ts = B0_T[pick]
        a = pd.concat([B0[(B0.variant == name) & (B0.t == tt)] for tt in ts[:60]]); b_ = pd.concat([B0[(B0.variant == "none") & (B0.t == tt)] for tt in ts[:60]])
        try: v.append(_sw(a) / _sw(b_))
        except Exception: pass
    lo, hi = np.nanpercentile(v, [2.5, 97.5]) if v else (np.nan, np.nan)
    say(f"  {name:22s} swing {_sw(g):5.2f}  ratio {_sw(g)/sw0:5.2f}  [{lo:.2f}, {hi:.2f}]   mean delta {g.delta.mean():+.2f}")

[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (B0 done) | server says: Ok

[B0] swing (right minus left delta, contrast >= 0.09, 20 images) for different prompt variants; ratio to no-history (3.15)
  none                   swing  3.15  ratio  1.00  [1.00, 1.00]   mean delta +0.18
  count_5_of_10          swing  1.37  ratio  0.44  [0.41, 0.46]   mean delta +0.04
  short_count            swing  2.22  ratio  0.71  [0.66, 0.77]   mean delta +0.38
  neutral_same_length    swing  3.55  ratio  1.13  [1.09, 1.18]   mean delta +0.28
  count_before_image     swing  0.64  ratio  0.20  [0.14, 0.25]   mean delta -1.14


In [9]:
# ===================== B1 / B1r: PATCHING (checkpointed) =====================
rng = np.random.default_rng(1)
ext = np.where(FULL & ((NR >= 8) | (NR <= 2)))[0]; rest = np.where(FULL & ~((NR >= 8) | (NR <= 2)))[0]
PATCH_T = np.sort(np.r_[rng.choice(ext, min(N_EXT, len(ext)), replace=False), rng.choice(rest, min(N_REST, len(rest)), replace=False)])
pp = f"{OUT}/patch_rows.csv"; done = pd.read_csv(pp) if os.path.exists(pp) else pd.DataFrame()
have = set(done.t) if len(done) else set(); rows = done.to_dict("records") if len(done) else []
t0 = time.time(); todo = [t for t in PATCH_T if t not in have]; print(f"patch trials: {len(PATCH_T)} | to do: {len(todo)} | forwards per trial: {2 + len(PATCH_LAYERS) * (2 if DO_REVERSE else 1)}")
for i, t in enumerate(todo):
    inn, inh = inputs(t, False, 0), inputs(t, True, 0)
    rec = dict(t=t, side=M.side.values[t], contrast=M.contrast.values[t], nr=int(NR[t]), d_none=RUN.delta(inn), d_hist=RUN.delta(inh))
    for L in PATCH_LAYERS:
        rec[f"fwd_{L}"] = RUN.delta(inn, edit=(L, "replace", HS[t, 1, L, :].astype(np.float32)))              # history state -> no-history run
        if DO_REVERSE: rec[f"rev_{L}"] = RUN.delta(inh, edit=(L, "replace", HS[t, 0, L, :].astype(np.float32)))   # no-history state -> history run
    rows.append(rec)
    if (i + 1) % 10 == 0 or i == len(todo) - 1:
        pd.DataFrame(rows).to_csv(pp, index=False); maybe_push('patching'); el = time.time() - t0
        print(f"  {i+1}/{len(todo)} | {el/60:.1f} min | ~{el/(i+1)*(len(todo)-i-1)/60:.0f} min left", flush=True)
PT = pd.DataFrame(rows).sort_values("t").reset_index(drop=True); PT.to_csv(pp, index=False); maybe_push('patching done', force=True)

patch trials: 32 | to do: 32 | forwards per trial: 8
  10/32 | 0.9 min | ~2 min left
  20/32 | 1.8 min | ~1 min left
  30/32 | 2.8 min | ~0 min left
  32/32 | 3.0 min | ~0 min left
[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (patching done) | server says: Ok


In [10]:
# ===================== B1 analysis =====================
e = PT[(PT.nr >= 8) | (PT.nr <= 2)].copy(); e["sgn"] = np.where(e.nr >= 8, 1.0, -1.0)
def ratio(df, col, base="d_none", full="d_hist", sub=False):
    num = (df.sgn * (df[col] - df[base])).mean() if not sub else (df.sgn * (df[full] - df[col])).mean()
    den = (df.sgn * (df[full] - df[base])).mean(); return num / den
def boot_ratio(df, col, sub=False, n=300, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        b = df.iloc[r.integers(0, len(df), len(df))]; v.append(ratio(b, col, sub=sub))
    return np.percentile(v, [2.5, 97.5])
rows_p = []
for L in PATCH_LAYERS:
    rec = dict(layer=L, recovery=ratio(e, f"fwd_{L}")); rec["rec_lo"], rec["rec_hi"] = boot_ratio(e, f"fwd_{L}")
    if DO_REVERSE: rec["removed"] = ratio(e, f"rev_{L}", sub=True); rec["rem_lo"], rec["rem_hi"] = boot_ratio(e, f"rev_{L}", sub=True)
    rows_p.append(rec)
P1 = pd.DataFrame(rows_p); P1.to_csv(f"{OUT}/b1_patch_summary.csv", index=False)
say(f"\n[B1] patching. trials with extreme counts: {len(e)}; mean history-induced shift (sign-aligned) = {(e.sgn * (e.d_hist - e.d_none)).mean():+.2f} logits")
say("recovery = fraction of the shift that returns when the history state replaces the no-history state at layer L (1 = all of it)")
say("removed  = fraction of the shift erased when the no-history state replaces the history state at layer L")
say(P1.round(2).to_string(index=False))
# swing (right minus left delta at contrast >= 0.09) under patching
hc = PT[PT.contrast >= 0.09]
def swing(df, col): return df[df.side == "right"][col].mean() - df[df.side == "left"][col].mean()
sw = dict(none=swing(hc, "d_none"), hist=swing(hc, "d_hist")); sw.update({f"fwd_{L}": swing(hc, f"fwd_{L}") for L in PATCH_LAYERS})
say(f"\n[C, patching] swing (contrast >= 0.09, n={len(hc)}): none {sw['none']:.2f} | hist {sw['hist']:.2f} (ratio {sw['hist']/sw['none']:.2f})")
say("  swing after patching the history state into the no-history run, ratio to none: " + ", ".join(f"L{L}: {sw[f'fwd_{L}']/sw['none']:.2f}" for L in PATCH_LAYERS))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].errorbar(P1.layer, P1.recovery, yerr=[P1.recovery - P1.rec_lo, P1.rec_hi - P1.recovery], marker="o", label="recovery (history -> none)")
if DO_REVERSE: ax[0].errorbar(P1.layer + .3, P1.removed, yerr=[P1.removed - P1.rem_lo, P1.rem_hi - P1.removed], marker="s", label="removed (none -> history)")
ax[0].axhline(1, c="gray", lw=.5); ax[0].axhline(0, c="gray", lw=.5); ax[0].set_xlabel("layer"); ax[0].set_title("B1: fraction of the prior shift carried by the last-position state"); ax[0].legend(fontsize=8)
ax[1].plot(PATCH_LAYERS, [sw[f"fwd_{L}"] / sw["none"] for L in PATCH_LAYERS], "o-", label="patched run"); ax[1].axhline(sw["hist"] / sw["none"], c="r", ls="--", label="history run"); ax[1].axhline(1, c="gray", lw=.5)
ax[1].set_xlabel("layer"); ax[1].set_title("C: swing ratio after patching (1 = no suppression)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/b1_patch.png", dpi=130); plt.close()


[B1] patching. trials with extreme counts: 24; mean history-induced shift (sign-aligned) = +0.21 logits
recovery = fraction of the shift that returns when the history state replaces the no-history state at layer L (1 = all of it)
removed  = fraction of the shift erased when the no-history state replaces the history state at layer L
 layer  recovery  rec_lo  rec_hi  removed  rem_lo  rem_hi
     8     -0.00   -0.18    0.11     0.11   -0.87    2.07
    12     -0.09   -2.71    0.70     0.09   -0.49    0.45
    16      0.39   -1.29    1.47     0.75   -0.91    2.66

[C, patching] swing (contrast >= 0.09, n=14): none 3.34 | hist 1.72 (ratio 0.51)
  swing after patching the history state into the no-history run, ratio to none: L8: 0.99, L12: 0.92, L16: 0.64


In [12]:
def _delta_fixed(self, inp, edit=None):
    handle = None
    if edit is not None:
        L, mode, vec = edit; v0 = torch.tensor(vec, dtype=torch.float16)
        def hook(mod, args, output):
            h = output[0] if isinstance(output, tuple) else output
            v = v0.to(h.device, h.dtype)                  # put the vector on the same GPU as this layer
            h = h.clone()
            if mode == "replace": h[:, -1, :] = v
            else: h[:, -1, :] = h[:, -1, :] + v
            return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
        handle = LAYERS[L - 1].register_forward_hook(hook)
    try:
        with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
    finally:
        if handle is not None: handle.remove()
    return float((lg[R_ID] - lg[L_ID]).item())
Runner.delta = _delta_fixed
RUN = Runner()

In [13]:
# ===================== B2 / C: STEERING a no-history run (checkpointed) =====================
rng = np.random.default_rng(2)
okc = np.where((M.contrast.values >= 0.0625))[0]
STEER_T = np.sort(np.r_[rng.choice(okc[M.side.values[okc] == "right"], N_STEER // 2, replace=False), rng.choice(okc[M.side.values[okc] == "left"], N_STEER // 2, replace=False)])
sp = f"{OUT}/steer_rows.csv"; sdone = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
srows = sdone.to_dict("records") if len(sdone) else []; seen = set(zip(sdone.layer, sdone.dir, sdone.s)) if len(sdone) else set()
base_inp = {t: inputs(t, False, 0) for t in STEER_T}
if not (len(sdone) and ("base", 0, 0) in seen):
    for t in STEER_T: srows.append(dict(layer=0, dir="base", s=0, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t])))
    pd.DataFrame(srows).to_csv(sp, index=False)
combos = [(L, d, s) for L in STEER_LAYERS for d in DIRS for s in (-1, 1) if (L, d, s) not in seen]
print(f"steering: {len(STEER_T)} images x {len(combos)} remaining (layer, direction, sign) combinations = {len(STEER_T)*len(combos)} forwards")
t0 = time.time()
for ci, (L, d, s) in enumerate(combos):
    vec = (s * DIRVEC[d][L]).astype(np.float32)
    for t in STEER_T: srows.append(dict(layer=L, dir=d, s=s, t=t, side=M.side.values[t], contrast=M.contrast.values[t], delta=RUN.delta(base_inp[t], edit=(L, "add", vec))))
    pd.DataFrame(srows).to_csv(sp, index=False); maybe_push('steering'); el = time.time() - t0
    print(f"  {ci+1}/{len(combos)} (L{L} {d} {s:+d}) | {el/60:.1f} min | ~{el/(ci+1)*(len(combos)-ci-1)/60:.0f} min left", flush=True)
ST = pd.DataFrame(srows)

steering: 16 images x 6 remaining (layer, direction, sign) combinations = 96 forwards
  1/6 (L14 shift -1) | 0.2 min | ~1 min left
  2/6 (L14 shift +1) | 0.4 min | ~1 min left
  3/6 (L14 evid -1) | 0.5 min | ~1 min left
  4/6 (L14 evid +1) | 0.7 min | ~0 min left
  5/6 (L14 rand1 -1) | 0.9 min | ~0 min left
  6/6 (L14 rand1 +1) | 1.1 min | ~0 min left


In [14]:
# ===================== B2 / C analysis =====================
base = ST[ST.dir == "base"].set_index("t").delta
def eff_boot(L, d, n=300, seed=0):
    a = ST[(ST.layer == L) & (ST.dir == d) & (ST.s == 1)].set_index("t").delta; b = ST[(ST.layer == L) & (ST.dir == d) & (ST.s == -1)].set_index("t").delta
    diff = ((a - b) / 2).values; r = np.random.default_rng(seed); bs = [diff[r.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return diff.mean(), *np.percentile(bs, [2.5, 97.5])
def swing_ratio(L, d, s):
    x = ST[(ST.layer == L) & (ST.dir == d) & (ST.s == s) & (ST.contrast >= 0.09)]; b0 = ST[(ST.dir == "base") & (ST.contrast >= 0.09)]
    return (x[x.side == "right"].delta.mean() - x[x.side == "left"].delta.mean()) / (b0[b0.side == "right"].delta.mean() - b0[b0.side == "left"].delta.mean())
rs = []
for L in STEER_LAYERS:
    for d in DIRS:
        m_, lo, hi = eff_boot(L, d); rs.append(dict(layer=L, dir=d, effect=m_, lo=lo, hi=hi, frac_of_natural=m_ / NAT_HALF, swing_ratio_plus=swing_ratio(L, d, 1), swing_ratio_minus=swing_ratio(L, d, -1)))
S2 = pd.DataFrame(rs); S2.to_csv(f"{OUT}/b2_steer_summary.csv", index=False)
say(f"\n[B2] steering. effect = (delta(+dir) - delta(-dir))/2 in logits, norm matched across directions; natural half-jump of the prior = {NAT_HALF:.2f} logits")
for L in STEER_LAYERS:
    say(f"  layer {L}: " + " | ".join(f"{d} {S2[(S2.layer == L) & (S2.dir == d)].effect.iloc[0]:+.2f}" for d in DIRS))
say("\nfull table (effect with 95% CI over images; frac_of_natural = effect / natural half-jump; swing ratios when adding +dir / -dir, 1 = no change):")
say(S2.round(2).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(12, 4)); cols = dict(zip(DIRS, ["C3", "C0", "C2", "C1", "gray", "silver"]))
for d in DIRS: x = S2[S2.dir == d]; ax[0].errorbar(x.layer, x.effect, yerr=[x.effect - x.lo, x.hi - x.effect], marker="o", label=d, color=cols[d], capsize=2)
ax[0].axhline(NAT_HALF, c="k", ls=":", label="natural half-jump"); ax[0].axhline(0, c="gray", lw=.5); ax[0].set_xlabel("layer"); ax[0].set_ylabel("effect on delta (logits)"); ax[0].legend(fontsize=7); ax[0].set_title("B2: steering at the answer position")
for d in DIRS: x = S2[S2.dir == d]; ax[1].plot(x.layer, (x.swing_ratio_plus + x.swing_ratio_minus) / 2, marker="o", label=d, color=cols[d])
ax[1].axhline(1, c="gray", lw=.5); ax[1].set_xlabel("layer"); ax[1].set_title("C: swing ratio under steering (1 = evidence untouched)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/b2_steer.png", dpi=130); plt.close()


[B2] steering. effect = (delta(+dir) - delta(-dir))/2 in logits, norm matched across directions; natural half-jump of the prior = 0.33 logits
  layer 10: shift -0.00 | evid +0.00 | rand1 +0.00
  layer 14: shift -0.10 | evid +0.23 | rand1 -0.01

full table (effect with 95% CI over images; frac_of_natural = effect / natural half-jump; swing ratios when adding +dir / -dir, 1 = no change):
 layer   dir  effect    lo    hi  frac_of_natural  swing_ratio_plus  swing_ratio_minus
    10 shift   -0.00 -0.00  0.00            -0.00              1.00               1.00
    10  evid    0.00 -0.00  0.00             0.00              1.00               1.00
    10 rand1    0.00  0.00  0.00             0.00              1.00               1.00
    14 shift   -0.10 -0.11 -0.08            -0.30              1.02               0.98
    14  evid    0.23  0.21  0.25             0.71              0.99               1.00
    14 rand1   -0.01 -0.01 -0.01            -0.04              1.00               1.00


In [15]:
# ===================== SUMMARY: paste everything printed by the previous cells back to Claude =====================
open(f"{OUT}/B_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push('final', force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")

[checkpoint] new version ponishbhatia3/llava-pve-b-ckpt (final) | server says: Ok
zip written: /kaggle/working/B_out.zip
